
#  School Data Cleaning & Preparation Project

## Author
Data Analyst Portfolio Project

## Business Understanding
Educational datasets often contain missing values, inconsistent formats, duplicated records, and data quality issues that can affect reporting and decision-making.

This project focuses on transforming raw school data into a clean and analysis-ready dataset suitable for SQL databases, Power BI dashboards, and reporting solutions.

---

#  Table of Contents

1. Project Overview
2. Data Loading
3. Data Understanding
4. Data Quality Assessment
5. Data Cleaning Process
6. Feature Engineering
7. Data Validation
8. SQL Preparation
9. Final Results


##  Data Loading

# Import core libraries used for data manipulation and numerical operations throughout the cleaning process.

In [15]:
import pandas as pd 
import numpy as np


##  Data Understanding

At this stage, the dataset structure is explored to understand:
- Available tables
- Number of records
- Column definitions
- Potential data quality issues


# Define the source Excel file path and retrieve all worksheet names to load the dataset dynamically.

In [16]:
file ="D:\depi\Powerpi\school_dataset.xlsx"

sheets = pd.ExcelFile(file).sheet_names

print(sheets)

['fact_student_assessments', 'dim_students', 'dim_teachers', 'dim_courses', 'dim_classes', 'dim_departments', 'dim_campuses', 'dim_terms', 'dim_guardians', 'README']


# Read every sheet from the Excel workbook and store each one in a dictionary for easy access.


##  Data Quality Assessment

The objective of this section is to identify:
- Missing values
- Invalid values
- Formatting inconsistencies
- Duplicate records


In [17]:
dfs={}
for sheet in sheets :
    dfs[sheet]=pd.read_excel(file,sheet_name=sheet)

# Quick inspection of the assessment fact table to understand the data structure and sample records.

In [18]:
dfs['fact_student_assessments'].head(10)

,record_id,learner_id,section_code,exam_dt,assessment_kind,score_points,max_mark,presence_flag,grade_txt
0,1,21021,CLS-0210,2024-04-25,Quiz,47.9,100,Present,C+
1,2,21086,CLS-0055,2024-10-05,Project,49.7,15,Present,D
2,3,20206,CLS-0049,2025-03-29,Quiz,75.4,20,Present,F
3,4,21637,CLS-0185,2024-05-21,Attendance,77.5,15,Present,C+
4,5,20631,CLS-0003,2024-02-18,Assignment,83.6,15,Present,D
5,6,20179,CLS-0022,2025-01-03,Assignment,71.8,50,Present,A
6,7,21668,CLS-0035,2025-01-04,Quiz,65.6,20,Absent,C
7,8,20888,CLS-0028,2024-10-27,Project,50.0,15,Present,B+
8,9,20342,CLS-0170,2024-12-17,Quiz,86.2,50,Present,B+
9,10,20440,CLS-0140,2024-07-13,Quiz,77.8,100,Present,D


# Standardize column names to match the data model and ensure consistency across related tables.

In [19]:
dfs['fact_student_assessments'].rename(columns={'section_code':'class_code'},inplace=True)
dfs['fact_student_assessments'].rename(columns={'learner_id':'student_key'},inplace=True)

# Review data types, null values, and overall structure before applying transformations.

In [20]:
dfs['fact_student_assessments'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13485 entries, 0 to 13484
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   record_id        13485 non-null  int64         
 1   student_key      13485 non-null  int64         
 2   class_code       13485 non-null  object        
 3   exam_dt          13485 non-null  datetime64[ns]
 4   assessment_kind  13485 non-null  object        
 5   score_points     13067 non-null  float64       
 6   max_mark         13485 non-null  int64         
 7   presence_flag    13485 non-null  object        
 8   grade_txt        13097 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(3), object(4)
memory usage: 948.3+ KB



##  Data Cleaning Process

Data cleaning operations are applied to improve data quality and ensure consistency across all tables.


# Create a fixed maximum score column to support percentage calculations.

In [21]:
dfs['fact_student_assessments']['max_mark']=100

# Calculate each student's assessment percentage based on the achieved score and maximum mark.

In [22]:
dfs['fact_student_assessments']['percentage'] = dfs['fact_student_assessments']['score_points'] / dfs['fact_student_assessments']['max_mark']

# Convert percentage scores into academic letter grades using predefined grading bands.

In [23]:
bins = [0, 0.5, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 1]
labels = ['F','D','C-','C','C+','B-','B','B+','A-','A']

dfs['fact_student_assessments']['grade_new'] = pd.cut(dfs['fact_student_assessments']['percentage'], bins=bins, labels=labels)

# Remove the original grade column after generating the standardized grading field.

In [24]:
dfs['fact_student_assessments'].drop('grade_txt',axis=1,inplace=True)

# Preview student dimension data before performing cleaning operations.

In [25]:
dfs['dim_students'].head(10)

,student_key,full_name,gender_code,birth_dt,campus_id,admission_year,scholarship_pct,student_mail,mobile_num,family_code
0,20001,Rana Samir,F,2013-08-23,CMP-02,2021.0,50.0,rana.samir896@schoolmail.edu,1638387225,FAM-2126
1,20002,Yara Fouad,female,2014-04-12,CMP-01,2024.0,25.0,yara.fouad873@schoolmail.edu,1962316484,FAM-1860
2,20003,Farah Adel,male,2016-10-17,CMP-03,NaN,NaN,farah.adel369@studentmail.com,1100923764,FAM-2130
3,20004,Layla Ashraf,F,2006-01-19,CMP-01,2023.0,0.0,layla.ashraf321@mail.edu,1989033742,FAM-2095
4,20005,Mariam Adel,F,2005-12-08,CMP-05,2022.0,15.0,mariam.adel757@studentmail.com,1659857207,FAM-2044
5,20006,Karim Khaled,F,2010-06-11,CMP-05,2025.0,0.0,karim.khaled98@schoolmail.edu,1409626069,FAM-1121
6,20007,Rana Gamal,M,2016-02-12,CMP-04,2019.0,0.0,rana.gamal10@schoolmail.edu,1806808156,FAM-1466
7,20008,Karim Ezz,male,2018-07-11,CMP-02,2024.0,NaN,karim.ezz680@schoolmail.edu,1023874165,FAM-2238
8,20009,Ali Tarek,NaN,2013-07-07,CMP-05,2025.0,50.0,ali.tarek175@mail.edu,1392010271,FAM-1330
9,20010,Nour Mahmoud,F,2016-03-06,CMP-03,2019.0,0.0,nour.mahmoud367@studentmail.com,1312517631,FAM-1087


# Standardize gender values into a consistent coded format (M/F).

In [26]:
dfs['dim_students']['gender_code']=dfs['dim_students']['gender_code'].replace({"male":"M" ,"female":"F"})

# Check and trim potential leading or trailing spaces in gender values.


##  Feature Engineering

Additional columns and transformations are created to improve downstream analysis and reporting.


In [28]:
dfs['dim_students']['gender_code'].str.strip()

0       F
1       F
2       M
3       F
4       F
       ..
1795    F
1796    M
1797    M
1798    F
1799    M
Name: gender_code, Length: 1800, dtype: object

# Replace missing gender records with a default 'unknown' category.

In [29]:
dfs['dim_students']['gender_code']=dfs['dim_students']['gender_code'].fillna("unknown")

# Audit missing student email addresses to determine the extent of data quality issues.

In [30]:
dfs['dim_students']['student_mail'].isna().sum()

np.int64(50)

# Generate institutional email addresses for students with missing email records.

In [33]:
student['student_mail2']=student['student_mail'].fillna(dfs['dim_students']['full_name'].str.lower()+dfs['dim_students']['student_key'].astype(str)+'@schoolmail.edu')

# Create a working copy/reference of the student dimension for focused transformations.

In [32]:
student=dfs['dim_students']

# Remove the original email column after creating the cleaned replacement field.


##  Data Validation

Validation checks are performed to confirm that cleaning operations were successfully applied.


In [34]:
student.drop(columns="student_mail",axis=1,inplace=True)

# Validate the student table structure after applying transformations.

In [35]:
student.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1800 entries, 0 to 1799
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   student_key      1800 non-null   int64         
 1   full_name        1800 non-null   object        
 2   gender_code      1800 non-null   object        
 3   birth_dt         1800 non-null   datetime64[ns]
 4   campus_id        1800 non-null   object        
 5   admission_year   1728 non-null   float64       
 6   scholarship_pct  1748 non-null   float64       
 7   mobile_num       1800 non-null   int64         
 8   family_code      1800 non-null   object        
 9   student_mail2    1800 non-null   object        
dtypes: datetime64[ns](1), float64(2), int64(2), object(5)
memory usage: 140.8+ KB


# Estimate missing admission years using birth year plus an assumed admission age.

In [36]:
student['admission_year']=student['admission_year'].fillna(student['birth_dt'].dt.year +18)

# Replace missing scholarship percentages with zero to avoid null values in analysis.

In [37]:
student['scholarship_pct']=student['scholarship_pct'].fillna(0)

# Create a working reference for the teacher dimension table.

In [38]:
teacher=dfs['dim_teachers']

# Review teacher table schema and data quality status.


##  SQL Preparation

The cleaned dataset is prepared for loading into SQL Server by defining data types and database mappings.


In [39]:
teacher.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   teacher_no      145 non-null    int64         
 1   teacher_name    145 non-null    object        
 2   dept_assigned   145 non-null    object        
 3   hire_date       145 non-null    datetime64[ns]
 4   monthly_salary  145 non-null    float64       
 5   campus_code     145 non-null    object        
 6   teacher_email   131 non-null    object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(4)
memory usage: 8.1+ KB


# Inspect sample teacher records before cleaning.

In [40]:
teacher.head(10)

,teacher_no,teacher_name,dept_assigned,hire_date,monthly_salary,campus_code,teacher_email
0,7001,Hassan Gamal,D107,2021-03-08,7736.65,CMP-03,hassan.gamal611@schoolmail.edu
1,7002,Layla Saber,D109,2023-09-19,5695.18,CMP-04,layla.saber354@studentmail.com
2,7003,Nada Gamal,D108,2019-12-31,28806.71,CMP-02,nada.gamal806@mail.edu
3,7004,Mariam Hamdy,D101,2024-06-21,8605.65,CMP-05,mariam.hamdy447@studentmail.com
4,7005,Karim Nabil,D110,2024-05-02,7465.62,CMP-02,karim.nabil524@mail.edu
5,7006,Mohamed Nabil,D109,2023-06-17,3690.59,CMP-01,mohamed.nabil395@schoolmail.edu
6,7007,Rana Mahmoud,D101,2017-12-14,14517.80,CMP-01,rana.mahmoud195@mail.edu
7,7008,Sara Mansour,D104,2016-03-23,6345.04,CMP-04,NaN
8,7009,Sara Ragab,D104,2019-03-14,10025.42,CMP-03,sara.ragab447@mail.edu
9,7010,Omar Ibrahim,D105,2024-08-22,15304.17,CMP-04,omar.ibrahim533@schoolmail.edu


# Generate default teacher email addresses where values are missing.

In [41]:
teacher['teacher_email']=teacher['teacher_email'].fillna(teacher['teacher_name'].str.lower()+teacher['teacher_no'].astype(str)+'@schoolmail.edu')

# Create a working reference for the courses dimension.

In [42]:
courses=dfs['dim_courses']

# Inspect course records and attributes.

In [43]:
courses.head(10)

,course_id,course_title,dept_link,credit_hours,difficulty_band
0,CRS001,Essentials of English,D111,3,Hard
1,CRS002,Intro to Statistics,D111,4,Medium
2,CRS003,Applied Biology,D106,2,Hard
3,CRS004,Advanced Chemistry,D111,3,Hard
4,CRS005,Lab in Physics,D107,3,NaN
5,CRS006,Foundations of Physics,D102,3,Hard
6,CRS007,Intro to Arabic,D100,5,Medium
7,CRS008,Essentials of History,D103,3,Medium
8,CRS009,Intro to Math,D101,4,Medium
9,CRS010,Intro to Biology,D107,4,NaN


# Review course table metadata and missing values.

In [44]:
courses.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 95 entries, 0 to 94
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   course_id        95 non-null     object
 1   course_title     95 non-null     object
 2   dept_link        95 non-null     object
 3   credit_hours     95 non-null     int64 
 4   difficulty_band  84 non-null     object
dtypes: int64(1), object(4)
memory usage: 3.8+ KB


# Fill missing course difficulty classifications with a default category.

In [45]:
courses['difficulty_band'].fillna("UnKnown",inplace=True)

C:\Users\rohaiem\AppData\Local\Temp\ipykernel_21456\3129396981.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  courses['difficulty_band'].fillna("UnKnown",inplace=True)


# Create a working reference for the classes dimension.

In [46]:
classes=dfs['dim_classes']

# Inspect class-related records and scheduling information.

In [47]:
classes.head(10)


,class_code,course_ref,teacher_ref,term_ref,room_label,meeting_days,start_time,capacity_limit
0,CLS-0001,CRS076,7118,T6,Lab2,Sun-Wed,11:00,30
1,CLS-0002,CRS020,7038,T8,B201,Mon-Thu,08:00,35
2,CLS-0003,CRS076,7046,T2,A101,Tue-Thu,15:30,30
3,CLS-0004,CRS029,7127,T7,Lab2,NaN,11:00,30
4,CLS-0005,CRS094,7071,T4,A101,Mon-Thu,11:00,20
5,CLS-0006,CRS084,7131,T3,NaN,Mon-Wed,08:00,30
6,CLS-0007,CRS068,7134,T8,Lab2,NaN,14:00,25
7,CLS-0008,CRS066,7013,T8,A101,Sun-Tue,11:00,35
8,CLS-0009,CRS047,7090,T7,B201,Tue-Thu,12:30,30
9,CLS-0010,CRS094,7112,T4,A101,NaN,11:00,35


# Review class table structure and data quality.

In [48]:
classes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 210 entries, 0 to 209
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   class_code      210 non-null    object
 1   course_ref      210 non-null    object
 2   teacher_ref     210 non-null    int64 
 3   term_ref        210 non-null    object
 4   room_label      189 non-null    object
 5   meeting_days    177 non-null    object
 6   start_time      210 non-null    object
 7   capacity_limit  210 non-null    int64 
dtypes: int64(2), object(6)
memory usage: 13.3+ KB


# Replace missing room labels with a default value.

In [49]:
classes['room_label'].fillna("UnKnown",inplace=True)


C:\Users\rohaiem\AppData\Local\Temp\ipykernel_21456\261315426.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  classes['room_label'].fillna("UnKnown",inplace=True)


# Replace missing meeting-day information with a default category.

In [50]:
classes['meeting_days'].fillna("UnKnown",inplace=True)

C:\Users\rohaiem\AppData\Local\Temp\ipykernel_21456\597102136.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  classes['meeting_days'].fillna("UnKnown",inplace=True)


# Create a working reference for the departments dimension.

In [51]:
department=dfs['dim_departments']

# Inspect department records and attributes.

In [52]:
department.head(10)


,dept_code,department_title,budget_band,active_flag
0,D100,Engineering Basics,Low,Y
1,D101,Music,Mid,Y
2,D102,Mathematics,Low,N
3,D103,Social Studies,High,Y
4,D104,PE,NaN,Y
5,D105,Languages,Low,Y
6,D106,Science,NaN,Y
7,D107,Support Services,Mid,N
8,D108,Arts,Mid,Y
9,D109,Business,Low,Y


# Review department table metadata and completeness.

In [53]:
department.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   dept_code         12 non-null     object
 1   department_title  12 non-null     object
 2   budget_band       10 non-null     object
 3   active_flag       12 non-null     object
dtypes: object(4)
memory usage: 516.0+ bytes


# Fill missing budget band values with a default category.

In [54]:
department['budget_band'].fillna("unknown",inplace=True)

C:\Users\rohaiem\AppData\Local\Temp\ipykernel_21456\1077048973.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  department['budget_band'].fillna("unknown",inplace=True)


# Load the remaining dimension tables required for the warehouse model.

In [55]:
campuses=dfs['dim_campuses']
terms=dfs['dim_terms']
guardians=dfs['dim_guardians']

# Preview campus records for validation purposes.

In [56]:
campuses.head(10)

,campus_ref,campus_name,zone_label,capacity_est
0,CMP-01,Nasr City,West,1376
1,CMP-02,Maadi,North,1969
2,CMP-03,6th October,East,1164
3,CMP-04,Alex West,North,1497
4,CMP-05,Sheikh Zayed,West,1963


# Review term table structure and data types.

In [57]:
terms.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   term_key   8 non-null      object
 1   term_name  8 non-null      object
 2   start_on   8 non-null      object
 3   end_on     8 non-null      object
 4   acad_year  8 non-null      object
dtypes: object(5)
memory usage: 452.0+ bytes


# Inspect academic term records.

In [68]:
terms.head()

,term_key,term_name,start_on,end_on,acad_year
0,T1,2023-Fall,2023-09-01,2023-12-20,2023/2024
1,T2,2023-Fall,2023-12-30,2024-04-18,2023/2024
2,T3,2024-Spring,2024-04-28,2024-08-16,2023/2024
3,T4,2024-Spring,2024-08-26,2024-12-14,2023/2024
4,T5,2024-Fall,2024-12-24,2025-04-13,2024/2025


# Review guardian table schema and missing values.

In [58]:
guardians.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2200 entries, 0 to 2199
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   guardian_ref    2200 non-null   int64  
 1   guardian_name   2200 non-null   object 
 2   relation_type   2116 non-null   object 
 3   guardian_phone  2160 non-null   float64
 4   occupation_txt  1954 non-null   object 
 5   household_ref   2200 non-null   object 
dtypes: float64(1), int64(1), object(4)
memory usage: 103.3+ KB


# Inspect guardian records before cleaning.

In [59]:
guardians.head(10)


,guardian_ref,guardian_name,relation_type,guardian_phone,occupation_txt,household_ref
0,90001,Karim Fouad,Sister,1.572706e+09,Teacher,FAM-2076
1,90002,Sara Fathy,NaN,1.192924e+09,Doctor,FAM-2092
2,90003,Rana Ashraf,Mother,1.310022e+09,Engineer,FAM-1360
3,90004,Nada Yassin,Father,1.560097e+09,Driver,FAM-2243
4,90005,Ahmed Ibrahim,Father,1.347728e+09,Sales,FAM-2009
5,90006,Nour Samir,Father,1.595047e+09,Teacher,FAM-1703
6,90007,Hassan Saad,Uncle,1.469981e+09,Eng.,FAM-1574
7,90008,Mostafa Saad,Mother,1.936089e+09,Doctor,FAM-2184
8,90009,Ahmed Fathy,Uncle,1.128155e+09,Doctor,FAM-2112
9,90010,Laila Saber,Sister,1.649464e+09,NaN,FAM-1687


# Replace missing guardian relationship types with a default category.

In [60]:
guardians['relation_type'].fillna("unknown",inplace=True)

C:\Users\rohaiem\AppData\Local\Temp\ipykernel_21456\4119736475.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  guardians['relation_type'].fillna("unknown",inplace=True)


# Replace missing guardian occupation values with a default category.

In [61]:
guardians['occupation_txt'].fillna("unknown",inplace=True)

C:\Users\rohaiem\AppData\Local\Temp\ipykernel_21456\3475507395.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  guardians['occupation_txt'].fillna("unknown",inplace=True)


# Install required database connectivity libraries for SQL Server integration.

In [62]:
pip install sqlalchemy pyodbc

Note: you may need to restart the kernel to use updated packages.


# Import SQLAlchemy and ODBC libraries to establish the database connection.

In [63]:
from sqlalchemy import create_engine,types as sqltypes

import urllib
import pyodbc

# Configure and create a SQL Server connection engine using Windows authentication.

In [65]:

params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=Localhost\SQLEXPRESS;"
    "DATABASE=school;"
    "Trusted_Connection=yes;"
)

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# Verify that the required ODBC drivers are available on the local machine.

In [66]:
import pyodbc

print(pyodbc.drivers())

['SQL Server', 'Oracle in OraDB21Home1', 'ODBC Driver 17 for SQL Server', 'ODBC Driver 18 for SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)']


# Import SQL data types that will be explicitly assigned during table creation.

In [72]:
from sqlalchemy.types import DateTime , String



# Define custom SQL data types for the assessment fact table.

In [70]:
dtype_map_fact = {
    'exam_dt': DateTime()
}

# Define SQL data types for student-related date and phone fields.

In [73]:
dtype_map_students = {
    'birth_dt': DateTime(),
    'mobile_num': String(20)
}

# Define SQL data types for teacher-related date fields.

In [74]:
dtype_map_teachers = {
    'hire_date': DateTime()
}

# Define SQL data types for academic term date fields.

In [75]:
dtype_map_terms = {
    'start_on': DateTime(),
    'end_on': DateTime()
}

# Define SQL data types for guardian phone number fields.

In [76]:
dtype_map_guardians = {
    'guardian_phone': String(20)
}

# Export all cleaned tables to SQL Server while preserving required column data types.

In [78]:

dtype_maps = {
    'fact_student_assessments': dtype_map_fact,
    'dim_students': dtype_map_students,
    'dim_teachers': dtype_map_teachers,
    'dim_terms': dtype_map_terms,
    'dim_guardians': dtype_map_guardians
}

for name, df in dfs.items():
   
    dtype_map = dtype_maps.get(name, {})  
    df.to_sql(
        name=name,
        con=engine,
        if_exists='replace',
        index=False,
        dtype=dtype_map,
        chunksize=1000
    )


# End of ETL process. All cleaned datasets have been loaded into the target database.


#  Project Summary

## Cleaning Activities Performed
- Missing value treatment
- Text standardization
- Data consistency improvements
- Column transformations
- SQL schema preparation

## Business Value
The final dataset is structured and reliable for:
- Power BI dashboards
- SQL reporting
- Data warehousing
- Analytical workflows

## Next Steps
- Load cleaned tables into SQL Server
- Build a star schema model
- Develop interactive Power BI dashboards
- Create KPI reporting layer
